In [ ]:
import time
import torch
import numpy as np
import gymnasium as gym

# --- Refactored Direct Imports (No 'old_muzero') ---
from modules.agent_nets.modular import ModularAgentNetwork
from agents.action_selectors.selectors import TemperatureSelector
from agents.action_selectors.policy_sources import SearchPolicySource

# MuZero specific imports
from search.search_py.modular_search import ModularSearch
from replay_buffers.buffer_factories import create_muzero_buffer
from replay_buffers.sequence import Sequence

# Learner imports
from agents.learner.base import UniversalLearner
from agents.learner.batch_iterators import (
    ReplayBufferIterator,
)  # Or your preferred iterator
from agents.registries.muzero import build_muzero

In [ ]:
# --- Explicit Hyperparameters ---
ENV_ID = "TicTacToe-v0"  # Replace with your specific Tic-Tac-Toe env ID
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# MuZero Core Params
NUM_SIMULATIONS = 25  # MCTS simulations per move
UNROLL_STEPS = 5  # Steps to unroll the world model in learning
TD_STEPS = 10  # N-step bootstrap target for value
DISCOUNT_FACTOR = 1.0  # Tic-Tac-Toe usually uses gamma=1.0
DIRICHLET_ALPHA = 0.3  # Root exploration noise
DIRICHLET_FRACTION = 0.25

# Training Params
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
TOTAL_EPISODES = 5000
TRAIN_STEPS_PER_EPISODE = 10

# --- Setup Environment & Dimensions ---
# Note: Ensure your env returns a 'legal_moves' list or mask in the info dict!
# For PettingZoo environments, you may need a wrapper to format the observation.
env = gym.make(ENV_ID)

# Typical Tic-Tac-Toe dimensions
obs_dim = env.observation_space.shape  # e.g., (3, 3, 3)
num_actions = env.action_space.n  # 9

print(f"Environment: {ENV_ID} | Obs Dim: {obs_dim} | Num Actions: {num_actions}")

In [ ]:
# --- 1. Agent Network (World Model) ---
# Your ModularAgentNetwork must be refactored to explicitly accept World Model kwargs
agent_network = ModularAgentNetwork(
    input_shape=obs_dim,
    num_actions=num_actions,
    use_world_model=True,
    representation_kwargs={"hidden_dim": 64},  # Explicit kwargs replacing config.arch
    dynamics_kwargs={"hidden_dim": 64},
    prediction_kwargs={"hidden_dim": 64},
).to(DEVICE)

# --- 2. Search Backend & Policy Source ---
search_engine = AOSSearch(
    num_simulations=NUM_SIMULATIONS,
    discount_factor=DISCOUNT_FACTOR,
    c_puct=1.25,
    dirichlet_alpha=DIRICHLET_ALPHA,
    dirichlet_fraction=DIRICHLET_FRACTION,
)

policy_source = SearchPolicySource(
    search_engine=search_engine,
    agent_network=agent_network,
    # Pass explicit search kwargs if your source needs them for temperature scaling
)

action_selector = TemperatureSelector(temperature=1.0)

# --- 3. Replay Buffer ---
replay_buffer = create_muzero_buffer(
    observation_dimensions=obs_dim,
    max_size=10000,  # Store up to 10k sequences/steps
    unroll_steps=UNROLL_STEPS,
    td_steps=TD_STEPS,
    gamma=DISCOUNT_FACTOR,
    num_actions=num_actions,
    observation_dtype=torch.float32,
)

# --- 4. Optimizer and Learner Components ---
optimizer = {
    "default": torch.optim.Adam(agent_network.parameters(), lr=LEARNING_RATE, eps=1e-5)
}

# Your registry must be refactored to accept kwargs instead of a config object
muzero_components = build_muzero(
    agent_network=agent_network,
    device=DEVICE,
    unroll_steps=UNROLL_STEPS,
    td_steps=TD_STEPS,
    discount_factor=DISCOUNT_FACTOR,
)

loss_pipeline = muzero_components["loss_pipeline"]
target_builder = muzero_components["target_builder"]

learner = UniversalLearner(
    agent_network=agent_network,
    device=DEVICE,
    num_actions=num_actions,
    observation_dimensions=obs_dim,
    observation_dtype=torch.float32,
    loss_pipeline=loss_pipeline,
    optimizer=optimizer,
    target_builder=target_builder,
    lr_scheduler={},
    callbacks=muzero_components.get("callbacks", []),
    clipnorm=5.0,
)

print("All components initialized successfully!")

In [ ]:
print("Starting explicit MuZero training loop...")

episodes_collected = 0

while episodes_collected < TOTAL_EPISODES:

    # ==========================================
    # PHASE 1: Self-Play Trajectory Collection
    # ==========================================
    # MuZero operates on full sequences rather than step-by-step transitions.
    sequence = Sequence(num_players=2)
    state, info = env.reset()

    # Track the current player (0 or 1). Depending on your env, you might
    # extract this from `info['player']` or `env.agent_selection`.
    current_player = 0

    sequence.append(
        observation=state,
        terminated=False,
        truncated=False,
        legal_moves=info.get("legal_moves", []),
    )

    done = False
    episode_reward = 0.0

    while not done:
        with torch.no_grad():
            # Format observation for the network
            obs_tensor = torch.tensor(
                state, dtype=torch.float32, device=DEVICE
            ).unsqueeze(0)

            # Format legal moves mask for MCTS
            legal_moves = info.get("legal_moves", np.arange(num_actions))
            mask = torch.zeros(num_actions, dtype=torch.bool, device=DEVICE)
            mask[legal_moves] = True
            info["legal_moves_mask"] = mask
            info["player"] = torch.tensor(
                [current_player], dtype=torch.int8, device=DEVICE
            )

            # 1. Run MCTS
            result = policy_source.get_inference(
                obs=obs_tensor,
                info=info,
                agent_network=agent_network,
                player_id=current_player,
                to_play=current_player,
            )

            # 2. Select Action (Using visit counts from MCTS)
            action, metadata = action_selector.select_action(
                result=result,
                info=info,
                episode_step=len(sequence),
            )

            action_val = action.item()

            # 3. Step Environment
            next_state, reward, terminated, truncated, next_info = env.step(action_val)
            done = terminated or truncated
            episode_reward += reward

            # 4. Store step in the Sequence
            # We save the MCTS visit counts as our 'target policy' and MCTS root value as 'target value'
            sequence.append(
                observation=next_state,
                terminated=terminated,
                truncated=truncated,
                action=action_val,
                reward=reward,
                policy=metadata.get("policy", result.probs).squeeze(0).cpu().numpy(),
                value=metadata.get("value", result.value).item(),
                player_id=current_player,
                legal_moves=next_info.get("legal_moves", []),
            )

            state = next_state
            info = next_info
            current_player = 1 - current_player  # Toggle player (if zero-sum 2-player)

    # Add terminal player state
    sequence.player_id_history.append(current_player)

    # Commit the sequence to the buffer
    replay_buffer.store_aggregate(sequence)
    episodes_collected += 1

    if episodes_collected % 10 == 0:
        print(
            f"Episode {episodes_collected} | Buffer Size: {replay_buffer.size} | Last Reward: {episode_reward}"
        )

    # ==========================================
    # PHASE 2: Optimization (Learning)
    # ==========================================
    # Only train if we have enough sequences
    if replay_buffer.size >= BATCH_SIZE:
        # Create an iterator to draw batches from the buffer
        iterator = ReplayBufferIterator(
            replay_buffer=replay_buffer,
            batch_size=BATCH_SIZE,
            num_batches=TRAIN_STEPS_PER_EPISODE,
            device=DEVICE,
        )

        for step_metrics in learner.step(iterator):
            # step_metrics contains loss values (value_loss, policy_loss, reward_loss)
            pass

print("MuZero Training complete!")